# Distilled ResNet-24 pipeline: teacher ensemble → student, held-out and all-50k

This notebook does what `train.sbatch` does, from a Jupyter session instead of a Slurm job: it picks the
flags, launches [`train.py`](train.py) once per stage, and streams each run's log into the cell.
`train.py` stays the single source of truth for the model, data and recipe.

The previous version distilled from one 18.8M-param teacher (97.1% local test) and the student matched it
(97.3%; Kaggle 0.897 at 200 epochs, 0.896 at 300), so the teacher is the ceiling. This version trains
`N_TEACHERS` copies of the teacher that differ only in their training seed (same data split, via
`--train-seed`) and the student matches the **average of their logits**, an ensemble that scores higher than
any single teacher. The student also gets random grayscale (`--grayscale 0.05`), the one measurable
difference between the Kaggle images and the training set (3% grayscale vs 1%).

The stages, in order:

| Stage | Run directory | What it is |
|---|---|---|
| 1 | `runs/teacher_w128`, `runs/teacher_w128_s1`, ... | `N_TEACHERS` 18.8M-param teachers, held-out split (40k / 5k / 5k), train seeds 0, 1, ... Over the 5M limit, **never submitted** |
| 2 | `runs/r24_kd_<TAG>` | 4.70M-param student distilled from the stage-1 ensemble. Its local test accuracy is the honest estimate |
| 3 | `runs/teacher_w128_full`, `runs/teacher_w128_s1_full`, ... | the same teachers trained on all 50,000 images |
| 4 | `runs/r24_kd_<TAG>_full` | the same student distilled from the stage-3 ensemble on all 50,000 images: **the submission** |

A stage whose `summary.json` already exists is skipped, so if the Jupyter session dies you just re-run
the notebook and it continues where it stopped. The existing `teacher_w128` and `teacher_w128_full` runs
are the seed-0 members of the ensembles and are reused as they are. If a teammate gives you a
`teacher_w128*/best.pt` plus its `summary.json`, drop them into `runs/` and that teacher is skipped too.

**On the NYU HPC**: open this through OOD / Jupyter on a **GPU node** with a kernel whose Python has
PyTorch, NumPy, Pillow and matplotlib (the project `.venv` or the `deep_learning` conda env). Run it with
the repo as the working directory. Rough A100 timings: each teacher 200 epochs ≈ 35 min, the student with
3 teachers ≈ 12 s/epoch (300 epochs ≈ 1 h). With the two seed-0 teachers already present, stages 1 + 2
≈ 2.2 h and stages 3 + 4 ≈ 2.2 h; ask for a session long enough or run the halves in separate sessions.


## 1. Setup

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / "train.py").exists(), f"run this notebook from the repo root, not {ROOT}"
sys.path.insert(0, str(ROOT))
import train          # noqa: E402  model and helpers from train.py
import model_summary  # noqa: E402  rules-mandated printout (params + test accuracy)

# Competition data: train_images/ + train_labels.csv, test_images/ + test.csv. The repo's data/ is used
# when it exists; otherwise point this at the copy on /scratch.
DATA_DIR = ROOT / "data" if (ROOT / "data" / "train_labels.csv").exists() else Path("/scratch/rl3725/deep_learning/project_1/data")
assert (DATA_DIR / "train_labels.csv").exists(), f"no competition data found in {DATA_DIR}"
RUNS = ROOT / "runs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # same rule as train.py's --device default
print("data dir", DATA_DIR)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

data dir /scratch/rl3725/deep_learning/project_1/data
python 3.12.16 | torch 2.15.0+cu132 | device cuda
gpu: NVIDIA A100-SXM4-80GB


## 2. Configuration

`EPOCHS` is the student's schedule and `N_TEACHERS` the ensemble size. Every key in the dicts below is a
`train.py` flag (`python train.py -h` lists them): a list becomes several values, `True` a bare flag,
`False`/`None` is omitted. The teachers are kept at 200 epochs so the existing `teacher_w128*` runs can be
reused as ensemble members; set `TEACHER_EPOCHS` and a new `TEACHER_NAME` to retrain them longer.

`SMOKE = True` runs every stage for 1 epoch on 1,000 images into `runs/smoke_pipeline/` (CPU is fine).
It only checks the plumbing.


In [ ]:
EPOCHS = 300               # student epochs (the 0.897 submission used 200; 300 scored the same)
N_TEACHERS = 3             # ensemble size; teacher k trains with --train-seed k on the shared split
TEACHER_EPOCHS = 200
TEACHER_NAME = "teacher_w128"   # teacher k is <TEACHER_NAME>_s<k> (k = 0 is plain <TEACHER_NAME>: the existing run)
SEED = 0                   # the data split, shared by every run here
GRAYSCALE = 0.05           # student only: probability that a training image is turned grayscale
SMOKE = False
SUFFIX = ""                # run-name suffix for a student variant (the teachers are reused as they are)
STUDENT_EXTRA = {}         # extra student flags, e.g. {"zoom": [0.7, 1.4]} or {"blocks": [4, 5, 3]}; overrides BASE/STUDENT keys
TAG = f"t{N_TEACHERS}_e{EPOCHS}" + (f"_{SUFFIX}" if SUFFIX else "")   # student runs are named r24_kd_<TAG> and r24_kd_<TAG>_full

# Shared by teacher and student: ResNet-24 (blocks 4 4 3), 3x3 convs, SGD recipe, Cutout + CutMix + label smoothing.
BASE = {
    "blocks": [4, 4, 3], "kernel": 3, "patience": 0,
    "batch": 128, "lr": 0.1, "wd": 5e-4, "warmup": 1.0,
    "jitter": [0.2, 0.2, 0.2], "label-smoothing": 0.1, "cutout": 16, "mix": "cutmix",
    "seed": SEED,
}
TEACHER = {"width": 128, "epochs": TEACHER_EPOCHS, "randaugment": [2, 9], "teacher-over-limit": True}  # 18,765,962 params
STUDENT = {"width": 64, "epochs": EPOCHS, "randaugment": [1, 9], "kd-alpha": 0.9, "kd-temp": 4,
           "grayscale": GRAYSCALE}                                                                   # 4,697,162 params
FULL = {"val-frac": 0, "test-frac": 0}   # train on all 50k, keep the last epoch

if SMOKE:
    RUNS = ROOT / "runs" / "smoke_pipeline"
teacher_names = [TEACHER_NAME if k == 0 else f"{TEACHER_NAME}_s{k}" for k in range(N_TEACHERS)]
TEACHERS = [(name, BASE | TEACHER | ({"train-seed": k} if k else {})) for k, name in enumerate(teacher_names)]
TEACHERS_FULL = [(f"{name}_full", cfg | FULL) for name, cfg in TEACHERS]
STUDENT_NAME, SUBMISSION_NAME = f"r24_kd_{TAG}", f"r24_kd_{TAG}_full"
STAGES = (
    TEACHERS
    + [(STUDENT_NAME, BASE | STUDENT | STUDENT_EXTRA | {"teacher": [RUNS / name / "best.pt" for name, _ in TEACHERS]})]
    + TEACHERS_FULL
    + [(SUBMISSION_NAME, BASE | STUDENT | STUDENT_EXTRA | FULL | {"teacher": [RUNS / name / "best.pt" for name, _ in TEACHERS_FULL]})]
)
HELD_OUT, ALL_50K = STAGES[:N_TEACHERS + 1], STAGES[N_TEACHERS + 1:]


def flags(cfg):
    """train.py command-line flags for a config dict."""
    out = []
    for k, v in cfg.items():
        if v is None or v is False:
            continue
        out.append(f"--{k}")
        if v is not True:
            out += [str(x) for x in (v if isinstance(v, list) else [v])]
    return out


def run_stage(name, cfg):
    """Run train.py for one stage, streaming its log; skipped when the run already has a summary.json."""
    out = RUNS / name
    if (out / "summary.json").exists():
        s = json.loads((out / "summary.json").read_text())
        print(f"[{name}] already done: {s['params']:,} params, {s['epochs_run']} epochs, "
              f"local test {s['test_acc']}, skipping")
        return out
    for teacher in cfg.get("teacher") or []:
        if not (Path(teacher).parent / "summary.json").exists():
            raise SystemExit(f"[{name}] needs a finished teacher run at {Path(teacher).parent}: run the earlier stage first")
    cmd = [sys.executable, "-u", str(ROOT / "train.py"), "--data", str(DATA_DIR), "--out", str(out)] + flags(cfg)
    if SMOKE:
        cmd.append("--smoke")
    print(f"[{name}] sbatch equivalent:  OUT={out.relative_to(ROOT)} ARGS=\"{' '.join(flags(cfg))}\"")
    print(f"[{name}] {' '.join(cmd)}\n", flush=True)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with open(out / "log.txt", "w") as log:
        proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        rc = proc.wait()
    print(f"\n[{name}] exit code {rc}  ({(time.time() - t0) / 60:.1f} min wall)")
    assert rc == 0, f"{name} failed, see log above or {out / 'log.txt'}"
    return out


for name, cfg in STAGES:
    print(f"{name:<28} {' '.join(flags(cfg))}")


## 3. Train

Held-out half first (the stage-1 teachers, then the stage-2 student) so the honest local test accuracy
exists before the submission run. Each cell blocks until its stages finish; the per-epoch log is the same
one a Slurm job writes.


In [ ]:
for name, cfg in HELD_OUT:
    run_stage(name, cfg)


In [ ]:
for name, cfg in ALL_50K:
    run_stage(name, cfg)


## 4. Results

The submitted model is stage 4, which has no local test score by construction (it trained on every
labeled image). Report stage 2's local test accuracy next to it; the two differ only in the data split.
The `teacher` column of the student rows is the ensemble's validation accuracy, with each member's in
brackets: the student can only be as good as that number lets it. Upload both `submission.csv` and
`submission_tta.csv` of stage 4 to Kaggle (the 0.897 was the TTA file).


In [ ]:
def fmt(v, spec=".4f"):
    return "n/a" if v is None else format(v, spec)

print(f"{'run':<28}{'params':>11}{'epochs':>8}{'best':>6}{'val':>8}{'test':>8}{'test+TTA':>10}{'min':>6}  tta changed  teacher (members)")
for name, _ in STAGES:
    p = RUNS / name / "summary.json"
    if not p.exists():
        print(f"{name:<28} not run"); continue
    s = json.loads(p.read_text())
    teacher = "" if s.get("teacher_val_acc") is None else \
        f"{s['teacher_val_acc']:.4f} ({', '.join(f'{a:.4f}' for a in s.get('teacher_val_accs') or [])})"
    print(f"{name:<28}{s['params']:>11,}{s['epochs_run']:>8}{s['best_epoch']:>6}{fmt(s['best_val_acc']):>8}"
          f"{fmt(s['test_acc']):>8}{fmt(s['test_acc_tta']):>10}{s['train_seconds'] / 60:>6.0f}  {s['kaggle_tta_changed']:<11}  {teacher}")

if (RUNS / SUBMISSION_NAME / "summary.json").exists():
    print(f"\nupload: {RUNS / SUBMISSION_NAME / 'submission_tta.csv'}  (or submission.csv without flip TTA)")


In [ ]:
# Required by the project rules: architecture table, trainable-parameter count against the 5M limit and the
# final test accuracy, for the submitted architecture (stage 2 is the held-out twin of the submitted run).
card = model_summary.report(run=RUNS / STUDENT_NAME)


In [ ]:
# Validation accuracy of the held-out student and its teachers, plus the training loss of the all-50k runs.
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linewidth": 0.6, "lines.linewidth": 1.6, "figure.dpi": 110})
hist = {name: json.loads((RUNS / name / "history.json").read_text())
        for name, _ in STAGES if (RUNS / name / "history.json").exists()}
col = lambda h, k: np.array([e[k] for e in h], dtype=float)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for name, _ in TEACHERS:
    if name in hist:
        a1.plot(col(hist[name], "epoch"), col(hist[name], "val_acc"), color=GRAY, alpha=0.6, label=name)
if STUDENT_NAME in hist:
    a1.plot(col(hist[STUDENT_NAME], "epoch"), col(hist[STUDENT_NAME], "val_acc"), color=BLUE, label=STUDENT_NAME)
a1.set(xlabel="epoch", ylabel="val accuracy", title="Held-out runs: validation accuracy", ylim=(0.8, 1))
for name, _ in TEACHERS_FULL:
    if name in hist:
        a2.plot(col(hist[name], "epoch"), col(hist[name], "train_loss"), color=GRAY, alpha=0.6, label=name)
if SUBMISSION_NAME in hist:
    a2.plot(col(hist[SUBMISSION_NAME], "epoch"), col(hist[SUBMISSION_NAME], "train_loss"), color=ORANGE, label=SUBMISSION_NAME)
a2.set(xlabel="epoch", ylabel="loss (vs mixed, smoothed targets)", title="All-50k runs: training loss")
for a in (a1, a2):
    a.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
